# SDE-Net / STGAN — quattro casi input–target, MAE e RMSE per bin

Analisi post-hoc delle previsioni salvate, senza training. Gli stessi cinque
bin percentuali e gli stessi tipi di figura di `pvgis_sde_pipeline` vengono
applicati ai quattro casi: normale→normale, normale→anomalo,
anomalo→normale e anomalo→anomalo, separatamente per t+1 e t+6.

“Target anomalo” indica l'etichetta STGAN del dato osservato da prevedere:
non è una classificazione della previsione del modello né una confusion matrix.

In [ ]:
import json, os, sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from physiq_pv.reporting.detector_threshold_sensitivity import load_prediction_errors
from physiq_pv.reporting.pointwise_detector_posthoc import detect_isolated_regional_solar_dropouts
from physiq_pv.reporting.posthoc_outputs import PERCENT_PRODUCTION_BINS
from physiq_pv.reporting.input_target_cases import (
    CASE_LABELS, load_clean_stgan_labels, classify_input_target_cases,
    summarize_input_target_cases, plot_input_target_cases,
)

## Regola esplicita per l'input

Configurazione iniziale: input anomalo se **almeno una delle 24 ore** della
sequenza esplicita è anomala STGAN, nella **stessa località del target**.
La finestra è `[issue_timestamp - 23h, issue_timestamp]`, inclusi gli estremi;
il target è `issue_timestamp + horizon_hours`. Questa è una scelta di analisi
configurabile tramite `INPUT_MIN_ANOMALOUS_STEPS`, non una regola del modello.
L'etichetta descrive la sequenza locale, non l'intero grafo spaziale né le ore
precedenti usate per costruire feature derivate.

Si usano anche le ore notturne per etichettare l'input; le metriche restano sui
target diurni (>10 W/m²), come nel report originale. Il top-1% STGAN viene
ricalcolato globalmente dopo il filtro qualità, prima di selezionare le finestre.
Una finestra con ore senza score o escluse dal controllo qualità non entra nei
quattro casi: viene esportata nell'audit. Anche un target senza etichetta è escluso.

In [ ]:
SDE_RUN = 'pvgis_stgnn_paper_faithful_gaussian_detector_mtgflow_ep60_h1-2-3-4-5-6_direct_seed1'
SDE_PREDICTIONS = Path(os.environ.get(
    'SDE_MULTIHORIZON_PREDICTIONS', ROOT / 'outputs' / SDE_RUN / 'predictions.csv'
)).resolve()
STGAN_SEED_DIR = Path(os.environ.get(
    'STGAN_SEED_DIR', ROOT / 'outputs/pvgis_stgan/paper_reference/seed_20'
)).resolve()
STGAN_SCORES = STGAN_SEED_DIR / 'anomaly_scores.csv'
REFERENCE_PEAK_CSV = Path(os.environ.get(
    'SDE_REFERENCE_PEAK_CSV', ROOT / 'outputs' / SDE_RUN / 'posthoc_by_horizon'
    / 't_plus_1' / 'reference_production_peaks.csv'
)).resolve()
PVGIS_2019 = Path(os.environ.get(
    'PVGIS_2019_FILE', os.environ.get('PVGIS_2019_PATH', ROOT / 'data/pvgis/piedmont_pvgis_2019.nc')
)).resolve()
STGAN_MANIFEST = Path(os.environ.get(
    'STGAN_PREPARED_MANIFEST', ROOT / 'outputs/pvgis_stgan/prepared/manifest.csv'
)).resolve()
PVGIS_QUALITY_SOURCE = Path(os.environ.get(
    'PVGIS_QUALITY_SOURCE', PVGIS_2019 if PVGIS_2019.is_file() else STGAN_MANIFEST
)).resolve()
OUT_DIR = Path(os.environ.get(
    'STGAN_INPUT_TARGET_OUT_DIR', ROOT / 'outputs/stgan_input_target_cases_t1_t6'
)).resolve()
HORIZONS = (1, 6)
SEQ_LEN = int(os.environ.get('SDE_INPUT_SEQ_LEN', '24'))
INPUT_MIN_ANOMALOUS_STEPS = int(os.environ.get('STGAN_INPUT_MIN_ANOMALOUS_STEPS', '1'))
CLEAN_TOP_K_PERCENT = 1.0
DAYTIME_THRESHOLD_WM2 = 10.0
print('Input:', SEQ_LEN, 'ore; anomalo con almeno', INPUT_MIN_ANOMALOUS_STEPS, 'ore anomale')
print('Output:', OUT_DIR)

In [ ]:
required = (SDE_PREDICTIONS, STGAN_SCORES, REFERENCE_PEAK_CSV, PVGIS_QUALITY_SOURCE)
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('File mancanti: ' + ', '.join(missing))
reference = pd.read_csv(REFERENCE_PEAK_CSV)
REFERENCE_PEAK_W = float(reference['reference_peak_w'].iloc[0])
if not np.isfinite(REFERENCE_PEAK_W) or REFERENCE_PEAK_W <= 0:
    raise ValueError('Picco di riferimento non valido.')
quality_issues = detect_isolated_regional_solar_dropouts(PVGIS_QUALITY_SOURCE)
labels = load_clean_stgan_labels(
    STGAN_SCORES, excluded_timestamps=quality_issues['timestamp'],
    top_percent=CLEAN_TOP_K_PERCENT,
)
errors = load_prediction_errors(
    SDE_PREDICTIONS, horizons=HORIZONS,
    daytime_threshold_wm2=DAYTIME_THRESHOLD_WM2,
    reference_peak_w=REFERENCE_PEAK_W, include_issue_timestamp=True,
)
print('Target diurni validi:', len(errors), '| Coordinate STGAN valide:', len(labels))

## Classificazione e copertura

I conteggi consentono di verificare quali finestre vengono escluse e quanto
supporto ha ciascun caso. Ogni previsione valida entra esattamente in un caso.
Le metriche sono aggregate su tutti i campioni del caso/bin/orizzonte;
non vengono calcolati MAE o RMSE giornalieri.

In [ ]:
classified = classify_input_target_cases(
    errors, labels, seq_len=SEQ_LEN, min_anomalous_steps=INPUT_MIN_ANOMALOUS_STEPS,
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
quality_issues.to_csv(OUT_DIR / 'pvgis_data_quality_issues.csv', index=False)
audit = classified.groupby(['horizon_hours', 'exclusion_reason'], dropna=False).size().rename('count').reset_index()
audit.to_csv(OUT_DIR / 'input_target_coverage_audit.csv', index=False)
audit_columns = [
    'location', 'timestamp', 'issue_timestamp', 'horizon_hours', 'production_bin',
    'input_scored_steps', 'input_anomalous_steps', 'input_anomaly_fraction',
    'input_is_anomaly', 'target_is_anomaly', 'case', 'exclusion_reason',
]
classified[audit_columns].to_csv(OUT_DIR / 'input_target_classification.csv', index=False)
classified.loc[classified['case'].isna(), audit_columns].to_csv(
    OUT_DIR / 'excluded_input_target_rows.csv', index=False,
)
display(audit)
if classified['case'].notna().sum() == 0:
    raise ValueError('Nessuna finestra completa: controllare copertura STGAN, seq_len e timestamp. Audit esportato.')
metrics = summarize_input_target_cases(classified, horizons=HORIZONS)
metrics.to_csv(OUT_DIR / 'input_target_bin_metrics.csv', index=False)
display(metrics[['horizon_hours', 'production_bin', 'case', 'count', 'mae', 'rmse']])
metadata = {
    'post_processing_only': True, 'training_rerun': False,
    'source_predictions': str(SDE_PREDICTIONS), 'stgan_scores': str(STGAN_SCORES),
    'horizons_hours': list(HORIZONS), 'seq_len': SEQ_LEN,
    'input_min_anomalous_steps': INPUT_MIN_ANOMALOUS_STEPS,
    'input_scope': 'same location, explicit sequence ending at issue_timestamp',
    'target_label': 'observed STGAN label, not a classification of the forecast',
    'clean_top_k_percent': CLEAN_TOP_K_PERCENT,
    'score_cutoff': labels.attrs['score_cutoff'],
    'quality_source': str(PVGIS_QUALITY_SOURCE),
    'quality_filter_policy': 'isolated_regional_solar_dropout_plus_immediate_recovery',
    'missing_input_policy': 'exclude any incomplete hourly window',
    'daytime_threshold_wm2': DAYTIME_THRESHOLD_WM2,
    'reference_peak_w': REFERENCE_PEAK_W, 'reference_peak_csv': str(REFERENCE_PEAK_CSV),
    'production_bins': [name for name, _, _ in PERCENT_PRODUCTION_BINS],
    'n_loaded_daytime_rows': len(errors),
    'n_classified_rows': int(classified['case'].notna().sum()),
    'n_excluded_rows': int(classified['case'].isna().sum()),
    'case_definitions': CASE_LABELS,
}
(OUT_DIR / 'analysis_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
del errors, labels, classified

## Stesse rappresentazioni della pipeline, per bin e per caso

Ogni bin e orizzonte ha due figure con i quattro casi affiancati:
**boxplot degli errori assoluti** (rombo rosso = MAE, linea = mediana;
baffi Tukey 1,5 IQR, outlier nascosti ma inclusi nel MAE) e
**barre dell'RMSE** aggregato. I casi senza campioni sono indicati con `n=0`
e metrica assente, mai con errore zero. Questi gruppi condizionano sia
l'input sia il target, quindi hanno un dominio più ristretto del report
che richiede soltanto l'etichetta del target.

In [ ]:
figures = plot_input_target_cases(metrics, OUT_DIR / 'figures')
for horizon in HORIZONS:
    display(Markdown(f'### Orizzonte t+{horizon}'))
    for bin_name, _, _ in PERCENT_PRODUCTION_BINS:
        display(Markdown(f'**{bin_name}**'))
        for metric in ('mae', 'rmse'):
            display(Image(filename=str(figures[f'{metric}_{bin_name}_t_plus_{horizon}'])))
print('Figure prodotte:', len(figures), '| Output:', OUT_DIR)